<a href="https://colab.research.google.com/github/deepan98raj-dotcom/My_Project/blob/main/Vector_%26_Semantic_Search_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Search Pipeline

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas numpy spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.8 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ==========================================
# STEP 1: Load Model & Sample Knowledge Base
# ==========================================
print("Loading SentenceTransformer model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Knowledge base data (Document collection)
documents = [
    {"id": 1, "text": "How do I reset my account password?", "category": "Account Management"},
    {"id": 2, "text": "I need help recovering my multi-factor authentication token.", "category": "Account Management"},
    {"id": 3, "text": "Unable to log into my user profile.", "category": "Account Management"},
    {"id": 4, "text": "The server crashed due to out of memory exception.", "category": "Tech Support"},
    {"id": 5, "text": "Database connections are timing out repeatedly.", "category": "Tech Support"},
    {"id": 6, "text": "API endpoint returning 500 internal server error.", "category": "Tech Support"},
    {"id": 7, "text": "Our team won the championship game last night!", "category": "Sports"},
    {"id": 8, "text": "What a stunning goal scored in the final minutes of the match!", "category": "Sports"},
    {"id": 9, "text": "The quarterback threw a 50 yard touchdown pass.", "category": "Sports"},
]

df = pd.DataFrame(documents)

# ==========================================
# STEP 2: Generate Vector Embeddings
# ==========================================
print("Generating document vector embeddings...")
embeddings = embedder.encode(df["text"].tolist(), show_progress_bar=False)

# Normalize vectors for Cosine Similarity scaling
embeddings = np.array(embeddings, dtype=np.float32)
faiss.normalize_L2(embeddings)

# ==========================================
# STEP 3: Create & Populate FAISS Index
# ==========================================
embedding_dim = embeddings.shape[1]  # 384 dimensions for all-MiniLM-L6-v2

# Using IndexFlatIP (Inner Product) for exact Cosine Similarity on normalized vectors
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print(f"FAISS Index successfully built with {index.ntotal} vectors of dimension {embedding_dim}.")

# ==========================================
# STEP 4: Real-Time Vector Search Function
# ==========================================
def search_vector_db(query: str, top_k: int = 3):
    """Encodes query, normalizes embedding, and searches FAISS index for top-K matches."""
    # 1. Encode & normalize query vector
    query_vector = embedder.encode([query])
    query_vector = np.array(query_vector, dtype=np.float32)
    faiss.normalize_L2(query_vector)

    # 2. Query FAISS index for top_k nearest neighbors
    scores, indices = index.search(query_vector, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        matched_doc = df.iloc[idx]
        results.append({
            "id": int(matched_doc["id"]),
            "text": matched_doc["text"],
            "category": matched_doc["category"],
            "similarity_score": round(float(score), 4),
        })

    return results

# ==========================================
# STEP 5: Run Sample Queries
# ==========================================
test_queries = [
    "I can't remember my login credentials",
    "Application server failure with memory limits",
    "Who scored the winning point in football?",
]

print("\n=== FAISS Vector Search Results ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    matches = search_vector_db(query, top_k=2)
    for i, match in enumerate(matches, start=1):
        print(f"  [{i}] Score: {match['similarity_score']} | Category: {match['category']}")
        print(f"      Text: '{match['text']}'")

Loading SentenceTransformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating document vector embeddings...
FAISS Index successfully built with 9 vectors of dimension 384.

=== FAISS Vector Search Results ===

Query: 'I can't remember my login credentials'
  [1] Score: 0.6664 | Category: Account Management
      Text: 'How do I reset my account password?'
  [2] Score: 0.6209 | Category: Account Management
      Text: 'Unable to log into my user profile.'

Query: 'Application server failure with memory limits'
  [1] Score: 0.6887 | Category: Tech Support
      Text: 'The server crashed due to out of memory exception.'
  [2] Score: 0.3857 | Category: Tech Support
      Text: 'Database connections are timing out repeatedly.'

Query: 'Who scored the winning point in football?'
  [1] Score: 0.508 | Category: Sports
      Text: 'What a stunning goal scored in the final minutes of the match!'
  [2] Score: 0.441 | Category: Sports
      Text: 'The quarterback threw a 50 yard touchdown pass.'


# Semantic Search Pipeline

In [ ]:
# Required packages
!pip install sentence-transformers faiss-cpu pandas numpy torch

In [ ]:
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

# Step 1: Initialize Dual Models (Bi-Encoder for Search, Cross-Encoder for Reranking)
print("Loading Embedding & Reranking Models...")
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Knowledge base simulating multi-domain enterprise docs
documents = [
    {"id": 101, "title": "Password Reset Guide", "text": "To reset your password, visit account.company.com/reset and follow the MFA prompts."},
    {"id": 102, "title": "Database Outage Troubleshooting", "text": "If connections pool times out, check PostgreSQL max_connections and increase backend memory limits."},
    {"id": 103, "title": "Sports Schedule", "text": "The corporate soccer league tournament final takes place this Saturday at 4 PM."},
    {"id": 104, "title": "VPN Connection Errors", "text": "Ensure your authentication token is synced when experiencing zero-trust network access drops."},
    {"id": 105, "title": "Database Optimization", "text": "Adding multi-column indexes on high-cardinality SQL queries reduces execution latency significantly."},
    {"id": 106, "title": "Account Lockout Policy", "text": "Accounts are locked for 30 minutes after 5 consecutive failed login attempts."},
]
df = pd.DataFrame(documents)

# Step 2: Generate Vectors & Build FAISS IVF Index (Approximate Nearest Neighbor)
embeddings = bi_encoder.encode(df["text"].tolist(), show_progress_bar=False)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]  # 384 dimensions
nlist = 2  # Number of Voronoi clusters (scaled higher for large datasets, e.g., 100+)

# Quantizer maps points to nearest cluster centroids
quantizer = faiss.IndexFlatIP(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

# Train quantizer clusters and add vectors
index.train(embeddings.astype(np.float32))
index.add(embeddings.astype(np.float32))

# Probe setting: controls trade-off between speed and recall accuracy
index.nprobe = 2  # Check top 2 nearest clusters during query time
print(f"IVFFlat Index built successfully. Total vectors indexed: {index.ntotal}\n")

# Step 3: Advanced Two-Stage Production Search Function
def advanced_semantic_search(query: str, top_k_retrieve: int = 4, top_n_rerank: int = 2):
    """
    Stage 1: Fast ANN vector search with FAISS (Bi-Encoder)
    Stage 2: High-accuracy deep score reranking (Cross-Encoder)
    """
    # --- STAGE 1: Candidate Retrieval ---
    query_vec = bi_encoder.encode([query])
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec.astype(np.float32), top_k_retrieve)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:  # Filter invalid index pads
            candidates.append(df.iloc[idx].to_dict())

    # --- STAGE 2: Cross-Encoder Reranking ---
    # Create input pairs: (query, candidate_document_text)
    pairs = [[query, doc["text"]] for doc in candidates]
    rerank_scores = cross_encoder.predict(pairs)

    # Attach rerank scores and sort candidates descending
    for i, score in enumerate(rerank_scores):
        candidates[i]["rerank_score"] = float(score)

    reranked_results = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return reranked_results[:top_n_rerank]

# Step 4: Run Real-Time Test Queries
queries = [
    "How to fix database taking too long to run queries?",
    "Why is my user profile blocked from logging in?"
]

print("=== Running Two-Stage Semantic Search Engine ===")
for q in queries:
    print(f"\nQuery: '{q}'")
    results = advanced_semantic_search(q, top_k_retrieve=4, top_n_rerank=2)
    for rank, res in enumerate(results, 1):
        print(f"  Rank {rank} [Score: {res['rerank_score']:.4f}] - {res['title']}")
        print(f"         Text: {res['text']}")

Loading Embedding & Reranking Models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

IVFFlat Index built successfully. Total vectors indexed: 6

=== Running Two-Stage Semantic Search Engine ===

Query: 'How to fix database taking too long to run queries?'
  Rank 1 [Score: -7.3511] - Database Optimization
         Text: Adding multi-column indexes on high-cardinality SQL queries reduces execution latency significantly.
  Rank 2 [Score: -8.9457] - Account Lockout Policy
         Text: Accounts are locked for 30 minutes after 5 consecutive failed login attempts.

Query: 'Why is my user profile blocked from logging in?'
  Rank 1 [Score: -9.3140] - Account Lockout Policy
         Text: Accounts are locked for 30 minutes after 5 consecutive failed login attempts.
  Rank 2 [Score: -11.3198] - VPN Connection Errors
         Text: Ensure your authentication token is synced when experiencing zero-trust network access drops.
